In [1]:
from datetime import date
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.io as pio
import pycountry

In [2]:
# Folder with WIA in Sharepoint
base_dir = "./"
data_in_folder = base_dir + "data_in/"
data_out_folder = base_dir + "data_out/"

# d&a climate folder
climate_dir = "d&a_climate/"

# out folder for maps (exposure)
maps_exposure_folder = base_dir + "data_out/maps/exposure/"

# shapes subfolder
data_in_shapes = base_dir + "data_in/shapes/"


In [3]:
# NOTE: selected countries for WIA execution
wia_countries_exercise = [
    # "Kenya",
    # "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    "Congo, The Democratic Republic of the",
    # "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
]
country = wia_countries_exercise[0]
c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3 if country != "Niger"
    # introduced Niger exception (known case to avoid Nigeria)
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)

# NOTE: select admin level and population year
admin_level = "2"
year = f"{date.today().year - 1}"

# Get csvs in climate for the country/adm
exposure_csvs = [
    f.name
    for f in Path(data_in_folder + climate_dir).iterdir()
    if f.name.endswith(".csv") and f"_{c_iso3.lower()}_adm{admin_level}" in f.name
]


In [4]:
# geojson output file name
geo_filename = f"geojson_{c_iso3}_adm{admin_level}"
# Get geojson in shapes
geojson_file = [
    f.name
    for f in Path(data_in_shapes).iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]

# read country shape geojson into gdf
gdf = gpd.read_file(
    data_in_shapes + f"{geojson_file[0]}"
).to_crs('EPSG:4326')

# column pcode and name (assume harmonized across CODs)
pcode_col = f"adm{admin_level}_pcode"
name_col = f"adm{admin_level}_name"


In [5]:
# concat exposure dataframes
exposure_df = pd.concat(
    [
        pd.read_csv(
            data_in_folder + climate_dir + csv_f
        ).astype({pcode_col: str})
        for csv_f in exposure_csvs
    ],
    ignore_index=True,
).sort_values(
    by=["hazard", pcode_col]
).merge(
    gdf[[pcode_col, name_col]],
    on=pcode_col,
    how="left",
).drop(columns="area")


In [6]:
# Check missing pcodes in exposure ouputs
# number of processed hazards
hazards = exposure_df.hazard.unique()
n_hazard = len(hazards)
if len(exposure_df) != len(gdf)*n_hazard:
    raise Exception(f"Stop execution: missing pcode(s) for selected {n_hazard} hazards")


In [7]:
# compute population rate in new column
# NOTE: save division by zero with fillna zeros
exposure_df[f"exposed_pop_share_{year}"] = (
    exposure_df["total_population_exposed"] / exposure_df["total_population"] * 100
).round(2).fillna(0)


In [8]:
# Disable automatic rendering
pio.renderers.default = None

for h in hazards:
    df = exposure_df.query("hazard == @h")

    # 0/1 binary mapping for wia: below or above mean/median
    exp_share_mean = df[f"exposed_pop_share_{year}"].mean()
    exp_share_med = df[f"exposed_pop_share_{year}"].median()

    # series mapping into new columns
    # NOTE: assumes exposed share has no nulls (should be 0 where no exposure)
    df["share_above_mean"] = (df[f"exposed_pop_share_{year}"] >= exp_share_mean).astype(int)
    df["share_above_med"] = (df[f"exposed_pop_share_{year}"] >= exp_share_med).astype(int)

    # save exposure individual data
    df[
        [
            pcode_col,
            name_col,
            f"exposed_pop_share_{year}",
            "share_above_mean",
            "share_above_med",
        ]
    ].to_excel(
        maps_exposure_folder + f"{c_iso3}_adm{admin_level}_{year}_{h}.xlsx",
        index=False,
        sheet_name=f"{c_iso3}_adm{admin_level}_{year}",
    )

    # map of hazard exposed population share by admin area polygon
    exposed_fig = px.choropleth(
        df,
        geojson=json.loads(gdf.to_json()),
        featureidkey=f"properties.{pcode_col}",
        locations=pcode_col,
        color=f"exposed_pop_share_{year}",
        color_continuous_scale="YlOrRd",
        projection="mercator",
        width=1200,
        height=900,
    )
    exposed_fig.update_geos(fitbounds="locations", visible=False)
    exposed_fig.update_layout(
        title=dict(
            text=f"Exposure {h} [%] ({c_iso3}_adm{admin_level})_{year}",
            x=0.5,  # x position (0-left, 1-right)
            y=0.97,  # y position (0-bottom, 1-top in normalized coordinates)
        ),
        margin={"r": 0, "t": 0, "l": 0, "b": 0}
    )
    exposed_fig.update_coloraxes(colorbar_len=0.65)
    exposed_fig.update_traces(marker_line_color="Gainsboro", marker_line_width=0.5)

    # write exposure figure as png locally
    exposed_fig.write_image(
        maps_exposure_folder + f"{c_iso3}_adm{admin_level}_{year}_{h}.png",
        format="png",
        scale=2,
    )
